# cc-1.4 — retail SFT **corpus / thought** inspection

Qualitative + quantitative look at the Stage-1.5 relabeled SFT corpora that feed
the `thoughts_*` Stage-2 arms. Each corpus is a pair of trajectory-list JSONs
(`train.json` / `eval.json`); every assistant message target is
`thought + "\n\n" + action` (see `scripts/export_sft_corpus.py`).

Corpora compared:
- `policy` vs `base` — thought scored by the **current LoRA** vs the **frozen
  base** model (Stage-1 `step_best` relabel).
- `*_last` — the same but relabeled from the Stage-1 `step_last` checkpoint.

These corpora are **regenerated by the running pipeline**; any that don't exist
yet are skipped with a printed note (the notebook never raises).

In [1]:
# === CONFIG =================================================================
from pathlib import Path  # (also imported in the SETUP cell below; safe)


def _repo_root(start=None):
    # Walk up to the repo root (dir with pyproject.toml + notebooks/) so paths
    # resolve whether launched from the repo root or from notebooks/.
    p = (Path(start) if start else Path.cwd()).resolve()
    for c in (p, *p.parents):
        if (c / "pyproject.toml").exists() and (c / "notebooks").exists():
            return c
    return Path.cwd()


REPO = _repo_root()
CORPUS_ROOT = REPO / "data" / "sft_corpus" / "tau2_retail"
CORPORA = ["policy", "base", "policy_last", "base_last"]

# how many chars to show per field in the printed examples
TRUNC = 240
N_EXAMPLES = 4
print("CORPUS_ROOT:", CORPUS_ROOT, "| exists:", CORPUS_ROOT.exists())

CORPUS_ROOT: /home/mzio/projects/act-prm-blog/data/sft_corpus/tau2_retail | exists: True


In [2]:
# === SETUP: headless backend + imports =======================================
import matplotlib
# Non-interactive rendering: the inline backend (Agg-based) is headless AND
# embeds figures as PNGs under nbconvert/jupyter; fall back to bare Agg if we
# are not running inside an IPython kernel.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    matplotlib.use("Agg")

import json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True,
                     "grid.alpha": 0.3, "figure.dpi": 110})

# Brand-neutral, colorblind-safe categorical palette.
PALETTE = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2"]


def load_metrics(mfile) -> pd.DataFrame:
    """Load a run's metrics.jsonl -> DataFrame, deduped on progress/batch
    (keep last: the eval flush writes a partial row before the merged one).
    Returns an empty frame (no exception) if the file is missing/empty."""
    mfile = Path(mfile)
    if not mfile.is_file():
        return pd.DataFrame()
    rows = []
    for line in mfile.read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    if "progress/batch" in df:
        df = (df.drop_duplicates("progress/batch", keep="last")
                .sort_values("progress/batch").reset_index(drop=True))
    return df


def _xy(df, col, xcol="progress/batch"):
    """(x, y) for a metric column, dropping rows where it is NaN/absent."""
    if df is None or df.empty or col not in df:
        return None, None
    keep = [c for c in (xcol, col) if c in df]
    sub = df[keep].dropna()
    if sub.empty:
        return None, None
    x = sub[xcol] if xcol in sub else np.arange(len(sub))
    return x, sub[col]


In [3]:
# === Loaders (graceful) =====================================================
def load_pool(path: Path):
    """Load one train/eval JSON (list of trajectory dicts). [] if missing/bad."""
    if not path.is_file():
        return None
    try:
        data = json.loads(path.read_text())
    except Exception as e:
        warnings.warn(f"could not parse {path}: {e}")
        return None
    return data if isinstance(data, list) else None


def split_thought_action(content: str):
    """Assistant target = 'thought\n\naction'. Split on the FIRST blank line."""
    if not isinstance(content, str):
        return "", ""
    parts = content.split("\n\n", 1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return "", content.strip()  # no thought (e.g. actions-only fallback)


def assistant_samples(traj: dict):
    """Yield (preceding_user_or_tool, thought, action, raw) per assistant msg."""
    msgs = traj.get("messages", []) if isinstance(traj, dict) else []
    prev = None
    for m in msgs:
        if not isinstance(m, dict):
            continue
        if m.get("role") == "assistant":
            th, ac = split_thought_action(m.get("content", ""))
            yield (prev, th, ac, m.get("content", ""))
        else:
            prev = m
            continue
        prev = None


def corpus_stats(name: str):
    """Return a dict of counts + thought lengths, or None if the corpus is absent."""
    cdir = CORPUS_ROOT / name
    tr = load_pool(cdir / "train.json")
    ev = load_pool(cdir / "eval.json")
    if tr is None and ev is None:
        return None
    th_words, th_chars, n_asst, n_with_thought = [], [], 0, 0
    for pool in (tr or [], ev or []):
        for traj in pool:
            for _prev, th, _ac, _raw in assistant_samples(traj):
                n_asst += 1
                if th:
                    n_with_thought += 1
                    th_words.append(len(th.split()))
                    th_chars.append(len(th))
    return {"name": name, "dir": cdir,
            "n_train_traj": None if tr is None else len(tr),
            "n_eval_traj": None if ev is None else len(ev),
            "n_assistant_samples": n_asst, "n_with_thought": n_with_thought,
            "thought_words": th_words, "thought_chars": th_chars,
            "train": tr, "eval": ev}

print("loaders ready")

loaders ready


In [4]:
# === Load all corpora (skip missing) ========================================
STATS = {}
for name in CORPORA:
    s = corpus_stats(name)
    if s is None:
        print(f"[skip] corpus not found yet: {CORPUS_ROOT / name}")
        continue
    STATS[name] = s
    tw = np.array(s["thought_words"]) if s["thought_words"] else np.array([])
    print(f"[ok]   {name:12s} train_traj={s['n_train_traj']} eval_traj={s['n_eval_traj']} "
          f"assistant_samples={s['n_assistant_samples']} with_thought={s['n_with_thought']} "
          f"med_thought_words={int(np.median(tw)) if tw.size else 0}")

if not STATS:
    print("\nNo SFT corpora present yet — they are being regenerated by the pipeline.")
    print("Re-run this notebook once data/sft_corpus/tau2_retail/*/{train,eval}.json exist.")

[skip] corpus not found yet: /home/mzio/projects/act-prm-blog/data/sft_corpus/tau2_retail/policy
[skip] corpus not found yet: /home/mzio/projects/act-prm-blog/data/sft_corpus/tau2_retail/base
[skip] corpus not found yet: /home/mzio/projects/act-prm-blog/data/sft_corpus/tau2_retail/policy_last
[skip] corpus not found yet: /home/mzio/projects/act-prm-blog/data/sft_corpus/tau2_retail/base_last

No SFT corpora present yet — they are being regenerated by the pipeline.
Re-run this notebook once data/sft_corpus/tau2_retail/*/{train,eval}.json exist.


## (a) Row counts per corpus

In [5]:
if STATS:
    tbl = pd.DataFrame([{
        "corpus": s["name"], "train_traj": s["n_train_traj"], "eval_traj": s["n_eval_traj"],
        "assistant_samples": s["n_assistant_samples"], "with_thought": s["n_with_thought"],
    } for s in STATS.values()])
    display(tbl)
else:
    print("no corpora to tabulate")

no corpora to tabulate


## (b) Thought-length distribution (words per thought)

In [6]:
if STATS:
    fig, ax = plt.subplots(figsize=(8, 4.4))
    allw = [w for s in STATS.values() for w in s["thought_words"]]
    if allw:
        hi = int(np.percentile(allw, 99)) + 1
        bins = np.linspace(0, max(hi, 1), 30)
        for i, (name, s) in enumerate(STATS.items()):
            if s["thought_words"]:
                ax.hist(s["thought_words"], bins=bins, alpha=0.5, label=name,
                        color=PALETTE[i % len(PALETTE)])
        ax.set(title="Thought length distribution", xlabel="words per thought",
               ylabel="count")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "no thoughts found (actions-only fallback?)",
                ha="center", va="center", transform=ax.transAxes, color="gray")
    plt.tight_layout(); plt.show()
else:
    print("no corpora to plot")

no corpora to plot


## (c) Example (state -> thought + action) samples

For a few trajectories we print the immediately preceding context message
(state), the inferred **thought**, and the committed **action** (all truncated).
Compare `policy` vs `base` (scorer) and `best` vs `_last` (EM checkpoint).

In [7]:
def trunc(x, n=TRUNC):
    x = "" if x is None else str(x)
    x = " ".join(x.split())
    return x if len(x) <= n else x[:n] + " …"


def show_examples(name, k=N_EXAMPLES):
    s = STATS.get(name)
    if s is None:
        print(f"[skip] {name}: not loaded"); return
    print("=" * 78)
    print(f"CORPUS: {name}   (dir: {s['dir']})")
    print("=" * 78)
    shown = 0
    for traj in (s["train"] or []):
        for prev, th, ac, _raw in assistant_samples(traj):
            if not (th or ac):
                continue
            state = prev.get("content") if isinstance(prev, dict) else None
            role = prev.get("role") if isinstance(prev, dict) else "?"
            print(f"\n--- example {shown + 1} (uid={traj.get('uid')}) ---")
            print(f"[state / prev {role}] {trunc(state)}")
            print(f"[thought]           {trunc(th)}")
            print(f"[action]            {trunc(ac)}")
            shown += 1
            if shown >= k:
                return
    if shown == 0:
        print("(no assistant samples found)")


if STATS:
    for name in STATS:
        show_examples(name)
else:
    print("no corpora to show")

no corpora to show


## Notes on the comparison

Once the corpora exist, read this section against the histogram + examples:

- **policy vs base scorer:** does the base model's scoring select systematically
  shorter / more literal thoughts than the on-policy LoRA? (Compare median
  thought-words and the examples.)
- **best vs `_last` checkpoint:** the `_last` corpora come from the fully-trained
  EM checkpoint; `cc-1.2` shows held-out reward *declined* past the peak, so
  `_last` thoughts may be more overfit to the training logs. Skim the examples
  for degenerate / templated thoughts.
- These qualitative reads feed the `cc-1.1` SFT comparison: whether inferred
  thoughts (and which scorer / checkpoint) recover the expert-thought lift over
  the `actions_only` baseline.